# Baseline
Here is provided the entire procedure of the baseline provided in the Common Code of the repository provided for the project

### Imports

In [ ]:
import torch
import cv2
import torchvision.models as models
import torch.nn as nn
import numpy as np
from transformers import pipeline, ViTModel
from PIL import Image
import matplotlib.pyplot as plt
import time
import torch.nn.functional as F
import segmentation_models_pytorch as smp

### Classifier
Since the required baseline should have a ViT as classifier, I took the one defined in ViT.py inside the Common Code folder of the original repository

In [ ]:
# class ROIClassifierViT(nn.Module):
#     def __init__(self, num_hoi_classes):
#         super().__init__()
        
#         # Channel reduction layers - same as ResNet version
#         self.first = nn.Conv2d(5, 4, kernel_size=1, stride=1, padding=0, bias=False)
#         self.pre_conv = nn.Conv2d(4, 3, kernel_size=1, stride=1, padding=0, bias=False)
        
#         # Vision Transformer backbone - using ViT-Base pretrained
#         self.backbone = ViTModel.from_pretrained('google/vit-base-patch16-224')
        
#         # Classification head - ViT outputs 768-dim features
#         self.fc = nn.Linear(768, num_hoi_classes)

        
        
#     def forward(self, x):
#         # Channel reduction: 5 -> 4 -> 3 channels
#         x = self.first(x)
#         x = self.pre_conv(x)
        
#         # ViT expects inputs in range [0, 1] and specific format
#         # Extract features using ViT backbone
#         outputs = self.backbone(pixel_values=x)
#         features = outputs.last_hidden_state[:, 0]  # Use [CLS] token representation
        
#         # Classification
#         out = F.sigmoid(self.fc(features))
#         return out


In [ ]:
class ROIClassifier(nn.Module):
    def __init__(self, num_hoi_classes):
        super().__init__()
      
        self.first = nn.Conv2d(5, 4, kernel_size=1, stride=1, padding=0, bias=False)
        self.pre_conv = nn.Conv2d(4, 3, kernel_size=1, stride=1, padding=0, bias=False)
        
        self.backbone = models.resnet18(pretrained=True)
        self.backbone.fc = nn.Identity()
        
        self.fc = nn.Linear(512, num_hoi_classes)
        
    def forward(self, x):
        
        x = self.first(x)
        x = self.pre_conv(x)         
        features = self.backbone(x)   
        out = F.sigmoid(self.fc(features))
        return out
    
class AutoEncoder(nn.Module):
    def __init__(self):
        super().__init__()
      
        self.encoder = nn.Sequential(
            nn.Conv2d(4, 4, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(4),
            nn.ReLU(),
            nn.Conv2d(4, 4, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(4),
            nn.ReLU(),
            nn.Conv2d(4, 4, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(4),
            nn.ReLU(),
            nn.Conv2d(4, 3, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(3),
            nn.ReLU(),
        )
        
        self.decoder = nn.Sequential(
            nn.Conv2d(3, 3, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(3),
            nn.ReLU(),
            nn.Conv2d(3, 3, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(3),
            nn.ReLU(),
            nn.Conv2d(3, 3, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(3),
            nn.ReLU(),
            nn.ConvTranspose2d(3, 4, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(4),
        )
        
        
    def forward(self, x):
        en = self.encoder(x)
        out = self.decoder(en)
  
        return out

### Functions
This part is about creating functions used inside the original baseline. These functions can be found inside the pipeline.py of the Common Code of the original github

Nel test_u_net.py l'immagine viene:

1. caricata;
2. convertita BGR → RGB;
3. ridimensionata a 256×256;
4. viene calcolata la depth;
5. RGB + depth diventano 4 canali;
6. viene normalizzata dividendo per 255.

In [ ]:
IMG_SIZE = (256, 256)
NUM_CLASSES = 22
BG_INDEX = NUM_CLASSES -1

tool_classes = list(range(0, 12))
tti_classes = list(range(12, 21))

In [ ]:
def load_unet_model(model_path, device):
    """
    Load the segmentation model used for semantic segmentation.
    """

    state = torch.load(model_path, map_location=device)

    if "model_state_dict" in state:
        state_dict = state["model_state_dict"]
    else:
        state_dict = state

    model = smp.Segformer(
        encoder_name="mit_b2",
        encoder_weights="imagenet",
        in_channels=4,
        classes=NUM_CLASSES
    )

    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()

    return model


def preprocess_unet_image(image, depth_model, device):
    """
    image: RGB numpy array HxWx3
    returns tensor 1x4x256x256
    """

    image_resized = cv2.resize(
        image,
        IMG_SIZE,
        interpolation=cv2.INTER_LINEAR
    )

    depth_map = np.array(
        depth_model(Image.fromarray(image_resized))["depth"]
    )

    image_4ch = np.concatenate(
        [
            image_resized,
            depth_map[..., None]
        ],
        axis=-1
    )

    tensor = (
        torch.from_numpy(image_4ch)
        .permute(2, 0, 1)
        .float()
        / 255.0
    )

    return tensor.unsqueeze(0).to(device)



def unet_inference(model, image, depth_model, device):
    """
    Returns a semantic segmentation map HxW
    containing class IDs.
    """

    input_tensor = preprocess_unet_image(
        image,
        depth_model,
        device
    )

    with torch.no_grad():
        prediction = model(input_tensor)

    segmentation = prediction.argmax(dim=1).squeeze(0)

    return segmentation.cpu().numpy().astype(np.uint8)


def parse_unet_output(segmentation):
    """
    Convert semantic segmentation map into the detection
    format expected by the rest of the pipeline.

    Returns:
        [
            {
                "class": int,
                "mask": np.ndarray
            },
            ...
        ]
    """

    detections = []

    for class_id in range(NUM_CLASSES - 1):

        mask = (segmentation == class_id).astype(np.uint8)

        if mask.sum() == 0:
            continue

        detections.append({
            "class": class_id,
            "mask": mask
        })

    return detections


Segmentation sarà qualcosa del tipo:

256 x 256

0 0 0 0 21 21 21

0 3 3 3 21 21 21

0 3 3 3 21 21 21

0 21 21 12 12 12

...

Dove
1. 0–11 = strumenti
2. 12–20 = TTI
3. 21 = background.


In [ ]:
def semantic_to_instances(segmentation, min_area=50):
    """
    Convert semantic segmentation into instance-like masks
    using connected components.
    """

    detections = []

    for class_id in range(NUM_CLASSES - 1):

        class_mask = (
            segmentation == class_id
        ).astype(np.uint8)

        if class_mask.sum() == 0:
            continue

        num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
            class_mask,
            connectivity=8
        )

        for component_id in range(1, num_labels):

            area = stats[component_id, cv2.CC_STAT_AREA]

            if area < min_area:
                continue

            instance_mask = (
                labels == component_id
            ).astype(np.uint8)

            detections.append({
                "class": class_id,
                "mask": instance_mask
            })

    return detections


In [ ]:
def found_objects(tool_list, tti_list, classes) -> list | list:
    
    tool_found = []
    tti_found = []
    
    for elem in range(len(classes)):
        if classes[elem] in tool_list:
            tool_found.append(elem)
        if classes[elem] in tti_list:
            tti_found.append(elem)
            
    return tool_found, tti_found

tool_classes = list(range(0, 12))

def find_tool_tissue_pairs(detections: list[dict]):
    '''
    incrocia ogni strumento rilevato con ogni tessuto 
    identificato per generare tutte le possibili coppie.
    '''
    
    tools = [d for d in detections if d['class'] in tool_classes]
    tissues = [d for d in detections if d['class'] not in tool_classes]
    pairs = []
    for s in tools:
        for o in tissues:
            pairs.append({'tool': s, 'tissue': o})
    return pairs


def extract_union_roi(image, tool_mask, tissue_mask, depth_map=None):
    '''
    Combina la maschera dello strumento con quella del tessuto per calcolare la regione rettangolare minima 
    (Bounding Box) che racchiude entrambi. Ritaglia l'immagine RGB e concatena ad essa sia la profondità che 
    le maschere fondi, creando un tensore a 5 canali (RGB + Depth + Mask)
    '''

    combined_mask = (tool_mask + tissue_mask).clip(0, 1).astype('uint8') #before astype
    x, y, w, h = cv2.boundingRect(combined_mask)
    roi = image[y:y+h, x:x+w]
  

    if depth_map is not None:
        depth_roi = depth_map[y:y+h, x:x+w]
        roi = np.concatenate([roi, depth_roi[..., None]], axis=-1)  # add depth as extra channel

    merged_mask = cv2.bitwise_or(tool_mask, tissue_mask)
    merged_mask = merged_mask[y:y+h, x:x+w]
    merged_mask = np.expand_dims(merged_mask, axis=-1)
    
    if merged_mask.shape[1] != roi.shape[1] or merged_mask.shape[0] != roi.shape[0]:
        print("MISMATCH")
        return None

    roi = np.concatenate([roi, merged_mask*255], axis=-1)

    return roi



def end_to_end_pipeline(image, yolo_model, depth_model, tti_classifier, device):
    # Step 1: YOLOv11-seg and depth estimation
  
    detections = yolo_inference(yolo_model, image)

    # depth_map = depth_model(image)
    depth_map = np.array(depth_model(image)["depth"])
    

    # depth_map = np.array(depth_model(Image.fromarray(image))["depth"])

    # Step 2: Pairing
    pairs = find_tool_tissue_pairs(detections)

    image = cv2.imread(image,cv2.IMREAD_COLOR)
   
    
    tti_predictions = []

    for pair in pairs:
        tool_mask = pair['tool']['mask']
        tissue_mask = pair['tissue']['mask']
        roi = extract_union_roi(image, tool_mask, tissue_mask, depth_map)
        if roi is None:
            return [] ,[]
        # Prepare input for ROI classifier
        roi_tensor = torch.from_numpy(roi).permute(2, 0, 1).unsqueeze(0).float() / 255.0
        roi_tensor = F.interpolate(roi_tensor, size=(224, 224), mode='bilinear', align_corners=False)
        roi_tensor = roi_tensor.to(device)
        
        # mean_rgb = [0.485, 0.456, 0.406]
        # std_rgb  = [0.229, 0.224, 0.225]
        # mean_t = torch.tensor(mean_rgb, device=device).view(1, 3, 1, 1)
        # std_t  = torch.tensor(std_rgb, device=device).view(1, 3, 1, 1)
        # roi_tensor[:, :3, :, :] = (roi_tensor[:, :3, :, :] - mean_t) / std_t
       
        tti_classifier.eval()
        with torch.no_grad():
            tti_logits = tti_classifier(roi_tensor)
            # tti_logits = torch.sigmoid(tti_classifier(roi_tensor))
            # print(tti_logits)
       
            # tti_logits = tti_logits.item()
            tti_class = torch.argmax(tti_logits, dim=1).item()
            tti_score = torch.softmax(tti_logits, dim=1).max().item()
            # if tti_logits >= 0.5:
            #     tti_class = 1
            # else:
            #     tti_class = 0
            # tti_score = tti_logits

        # Save ROI result
        tti_predictions.append({
            'tool': pair['tool'],
            'tissue': pair['tissue'],
            'tti_class': tti_class,
            'tti_score': tti_score
        })

    return detections, tti_predictions


def show_mask_overlay_from_binary_mask(image_bgr, binary_mask, alpha=0.5, mask_color=(1.0, 0.0, 0.0)):

    
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

    colored_mask = np.zeros_like(image_rgb)
    colored_mask[..., 0] = mask_color[0]  
    colored_mask[..., 1] = mask_color[1]  
    colored_mask[..., 2] = mask_color[2]  


    overlay = image_rgb.copy()
    indices = binary_mask.astype(bool)
    overlay[indices] = (1 - alpha) * image_rgb[indices] + alpha * colored_mask[indices]

    plt.figure(figsize=(8, 8))
    plt.imshow(overlay)
    plt.axis('off')
    plt.show()
    return cv2.cvtColor((overlay * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)

### Model loading

In [ ]:
model = load_yolo_model('outputs/yolo26/train_yolo26/weights/best.pt')
pipe = pipeline(task="depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

### Image directory

In [ ]:
image = 'yolo_dataset/images/test/adnansetlc37005_frame000019.png'

1. Classification with neural network

```text
         immagine
            ↓
         YOLO segmentation
            ↓
         tool + tissue detection
            ↓
         creazione delle coppie tool/tissue
            ↓
         calcolo depth map
            ↓
         ROI = RGB + depth + mask
            ↓
         ROIClassifier
            ↓
         TTI class
```
And this is the version of the pipeline.py of the original github.

In [ ]:
tti_class = ROIClassifier(2)
tti_class.load_state_dict(torch.load('ROImodel.pt',map_location=device))
tti_class.to(device)

# tti_class = ROIClassifierViT(2)
# tti_class.load_state_dict(torch.load('Vit2.pt',map_location=device))
# tti_class.to(device)

In [ ]:
detection , tti_predictions  = end_to_end_pipeline(image,model,pipe,tti_class,device)

print()

# print(tti_predictions)

image_full = cv2.imread(image, cv2.IMREAD_COLOR)


H_full, W_full = image_full.shape[:2]

for i in range(len(tti_predictions)):
    tool_mask_full   = tti_predictions[i]['tool']['mask']    
    tissue_mask_full = tti_predictions[i]['tissue']['mask'] 

    # print("TOOL: " , tti_predictions[i]['tool']['class'])
    # print("TISSUE: " , tti_predictions[i]['tissue']['class'])
    print("TTI: " , tti_predictions[i]['tti_class'])

    tool_mask_resized = cv2.resize(
        tool_mask_full.astype(np.uint8),
        (W_full, H_full),
        interpolation=cv2.INTER_NEAREST
    )
    tissue_mask_resized = cv2.resize(
        tissue_mask_full.astype(np.uint8),
        (W_full, H_full),
        interpolation=cv2.INTER_NEAREST
    )


    show_mask_overlay_from_binary_mask(image_full, tool_mask_resized, mask_color=(1.0, 0.0, 0.0))

    show_mask_overlay_from_binary_mask(image_full, tissue_mask_resized, mask_color=(0.0, 1.0, 0.0))
